# TurnWave Phase 2+3 — acoustic branch and fusion (Colab)

**Before running:** Runtime -> Change runtime type -> **T4 GPU** -> Save. Then Runtime -> Run all.

Roughly: 25 min to build the feature cache, 25 min audio training, 10 min fusion, then export.

In [ ]:
# 1. Setup. Asserts loudly rather than failing three cells later.
import os, subprocess, sys

if not os.path.isdir('/content/turnwave'):
    !git clone -q https://github.com/Nikhils-G/turnwave.git /content/turnwave
%cd /content/turnwave
!git pull -q

install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.',
                          '--no-deps', 'sentencepiece', 'datasets', 'soundfile',
                          'onnx', 'onnxruntime', 'onnxscript'])
assert install.returncode == 0, 'install failed - read the error above'

import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU.'
print('GPU OK:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Phase 1 text branch: dataset, tokenizer, training. Needed for fusion.
# Skip the training line if you already have checkpoints/text_eot/best.pt uploaded.
!python scripts/build_text_dataset.py --out data/text
!python -m turnwave.tokenizer data/text/corpus.txt checkpoints/tokenizer
!python -m turnwave.train --task text \
    --train data/text/train.jsonl --val data/text/validation.jsonl \
    --tokenizer checkpoints/tokenizer/spm.model --out checkpoints/text_eot \
    --steps 3500 --batch-size 256 --num-workers 2

In [ ]:
# 3. Stream the audio corpus and precompute the log-mel cache (~20-30 min).
# One row yields one example per pause: the final pause ended the turn, the
# earlier ones did not. Features are computed once and reused every epoch.
!python scripts/build_audio_dataset.py --out data/audio \
    --max-examples 60000 --max-eval-examples 6000

In [ ]:
# 4. Train the acoustic branch (~25 min).
!python -m turnwave.train --task audio --cache data/audio \
    --out checkpoints/audio_eot \
    --steps 4000 --batch-size 128 --lr 3e-4 --num-workers 2

In [ ]:
# 5. Train the fusion head over both frozen branches (~10 min).
!python -m turnwave.train --task fusion --cache data/audio \
    --tokenizer checkpoints/tokenizer/spm.model \
    --text-ckpt checkpoints/text_eot/best.pt \
    --audio-ckpt checkpoints/audio_eot/best.pt \
    --out checkpoints/fusion_eot \
    --steps 2000 --batch-size 128 --lr 1e-3 --num-workers 2

In [ ]:
# 6. THE ABLATION: text vs audio vs fused, on one held-out set.
!python -m turnwave.ablate --cache data/audio \
    --tokenizer checkpoints/tokenizer/spm.model \
    --text-ckpt checkpoints/text_eot/best.pt \
    --audio-ckpt checkpoints/audio_eot/best.pt \
    --fusion-ckpt checkpoints/fusion_eot/best.pt \
    --split test --device cuda

In [ ]:
# 7. Export to ONNX and measure CPU latency. Reports whether INT8 actually
# helps for each branch -- it speeds up the transformer and slows the CNN.
!python -m turnwave.export --ckpt checkpoints/text_eot/best.pt --out-dir checkpoints/onnx
!python -m turnwave.export --ckpt checkpoints/audio_eot/best.pt --out-dir checkpoints/onnx
!python -m turnwave.export --ckpt checkpoints/fusion_eot/best.pt --out-dir checkpoints/onnx

In [ ]:
# 8. Training curves for the README.
!python scripts/plot_training.py checkpoints/audio_eot/log.csv --out docs/audio_curves.png
!python scripts/plot_training.py checkpoints/fusion_eot/log.csv --out docs/fusion_curves.png
from IPython.display import Image, display
display(Image('docs/audio_curves.png'), Image('docs/fusion_curves.png'))

In [ ]:
# 9. Save everything before the session expires.
!zip -qr turnwave_phase2.zip checkpoints/audio_eot checkpoints/fusion_eot \
    checkpoints/onnx docs/audio_curves.png docs/fusion_curves.png
from google.colab import files
files.download('turnwave_phase2.zip')